<a href="https://colab.research.google.com/github/alltimerookie-og/moodly/blob/main/moodly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers joblib "pydantic-ai-slim[groq]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 kB 5.4 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer
import joblib
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# 1. Load your saved models
embedder = SentenceTransformer('all-MiniLM-L6-v2') # Don't change
clf = joblib.load('/content/drive/MyDrive/Hackathons/HackForHumanity/Models/mental_health_xgboost_CALIBRATED.pkl') # Change path from the Models to PickleFiles folder
label_encoder = joblib.load('/content/drive/MyDrive/Hackathons/HackForHumanity/Models/label_encoder.pkl') # Change path from the Models to PickleFiles folder

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/usr/lib/python3.13/pickle.py:1754: UserWarning: [20:40:20] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


In [6]:
import json

def get_agent_directive(text, crisis_threshold=15.0): # Updated threshold to 10.0 (meaning 10%)
    # 1. Get raw probabilities
    embedding = embedder.encode([text])
    raw_probabilities = clf.predict_proba(embedding)[0]
    class_names = label_encoder.classes_

    # ✅ Multiply by 100 here to shift to a 0-100 scale
    status_dict = {name: float(prob) * 100 for name, prob in zip(class_names, raw_probabilities)}

    # 2. Crisis check (now compares against a 0-100 scale)
    suicidal_prob = status_dict.get('Suicidal', 0.0)
    is_crisis = bool(suicidal_prob >= crisis_threshold)

    # 3. Primary tone
    sorted_statuses = dict(sorted(status_dict.items(), key=lambda x: x[1], reverse=True))
    primary_tone = list(sorted_statuses.keys())[0]

    # 4. Payload (rounded to 1 decimal place for cleaner JSON)
    payload = {
        "is_crisis": is_crisis,
        "primary_tone": primary_tone,
        "suicidal_probability": round(suicidal_prob, 1),
        "all_probabilities": {k: round(v, 1) for k, v in sorted_statuses.items()},
        "agent_directive": ""
    }

    # 5. Directives
    if is_crisis:
        payload["agent_directive"] = "CRITICAL SAFETY OVERRIDE: The user is expressing suicidal ideation. STOP casual companionship. Validate their pain, do not give medical advice, and immediately provide crisis hotline numbers (e.g., 988)."
    elif primary_tone == "Anxiety":
        payload["agent_directive"] = "The user is anxious. Use a calm, grounding, and reassuring tone. Keep sentences relatively short and ask low-pressure questions."
    elif primary_tone == "Depression":
        payload["agent_directive"] = "The user is experiencing low mood/depression. Be gentle, empathetic, and avoid toxic positivity. Let them lead the conversation."
    else:
        payload["agent_directive"] = f"Adopt a conversational tone suitable for a user feeling '{primary_tone}'."

    return json.dumps(payload, indent=2)
# ---------------------------------------------------------
# Test it with your exact phrase!
# ---------------------------------------------------------
user_text = "i wanna give up, I'm feeling anxious"
agent_payload = get_agent_directive(user_text)
print(agent_payload)

{
  "is_crisis": true,
  "primary_tone": "Anxiety",
  "suicidal_probability": 16.5,
  "all_probabilities": {
    "Anxiety": 65.4,
    "Suicidal": 16.5,
    "Depression": 9.7,
    "Normal": 7.7,
    "Stress": 0.5,
    "Personality disorder": 0.1,
    "Bipolar": 0.1
  },
  "agent_directive": "CRITICAL SAFETY OVERRIDE: The user is expressing suicidal ideation. STOP casual companionship. Validate their pain, do not give medical advice, and immediately provide crisis hotline numbers (e.g., 988)."
}


In [7]:
import os
import getpass

# Securely input your API key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("🔑 Paste your Groq API Key here: ")
    print("✅ API Key set successfully!")

🔑 Paste your Groq API Key here: ··········
✅ API Key set successfully!


In [8]:
import os
import asyncio
from typing import Any

from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.groq import GroqModel
from pydantic_ai.providers.groq import GroqProvider

In [9]:

# ==========================================
# 1. Pydantic Models (The Strict Contract)
# ==========================================
class EmotionalState(BaseModel):
    is_crisis: bool
    primary_tone: str
    suicidal_probability: float
    state_deltas: dict[str, float] = Field(description="Change in probabilities since last message. Positive means it increased.")
    current_state: dict[str, float]

class AgentResponse(BaseModel):
    internal_monologue: str = Field(description="Your private reasoning. Analyze the state_deltas. Did your last message make things better or worse? How will you adjust?")
    safety_override_active: bool = Field(description="True if is_crisis is True in the EmotionalState.")
    message_to_user: str = Field(description="The actual empathetic response to the user. Keep it under 3 sentences.")

# ==========================================
# 2. Dependencies (Only for Session State)
# ==========================================
class MentalHealthDeps(BaseModel):
    previous_state: dict[str, float] | None = None
    user_profile: dict = Field(default_factory=lambda: {
        "effective_strategies": ["grounding exercises", "silent listening"],
        "ineffective_strategies": ["complex advice", "toxic positivity"]
    })

In [10]:
# ==========================================
# 3. Initialize the Groq Model & Agent
# ==========================================
# Per PydanticAI docs: Explicitly initialize the Groq model
groq_provider = GroqProvider(api_key=os.environ["GROQ_API_KEY"])
groq_model = GroqModel('openai/gpt-oss-120b', provider=groq_provider)


agent = Agent(
    model=groq_model,
    deps_type=MentalHealthDeps,
    output_type=AgentResponse, # Forces the LLM to output the Pydantic model
    system_prompt="""
You are an empathetic mental health companion AI.
You are NOT a doctor. Never diagnose or prescribe.

**YOUR GOAL:** Lower the user's negative state scores over time.

**HOW TO USE THE TOOL:**
Before replying, you MUST call the `analyze_emotional_state` tool with the user's latest message.

**SELF-IMPROVEMENT RULES:**
Look at the `state_deltas` returned by the tool.
- If negative scores (Anxiety, Suicidal, Depression) INCREASED (positive delta), your previous approach failed. Acknowledge this in your `internal_monologue` and pivot to a lower-demand, validating approach.
- If negative scores DECREASED (negative delta), reinforce your current tone.
- Use the `user_profile` to avoid strategies that have historically failed for this user.

**SAFETY PROTOCOL:**
If the tool returns `is_crisis: True`, you MUST set `safety_override_active: True` and your `message_to_user` MUST be the exact safety script below. Do not deviate.
Safety Script: "I hear how much pain you're in, and I'm really glad you told me. Because your safety is my absolute top priority, I need to ask: Are you thinking about hurting yourself right now? If you are, please call or text the Suicide & Crisis Lifeline at 988 immediately. I am here with you."
"""
)

# ==========================================
# 4. The ML Sensor Tool
# ==========================================
@agent.tool
async def analyze_emotional_state(ctx: RunContext[MentalHealthDeps], text: str) -> EmotionalState:
    """Analyzes the user's emotional state and calculates changes from the previous message."""
    deps = ctx.deps

    # Use the globally loaded models directly
    # asyncio.to_thread prevents CPU-bound ML tasks from blocking the async Groq loop
    embedding = await asyncio.to_thread(embedder.encode, [text])
    raw_probabilities = await asyncio.to_thread(clf.predict_proba, embedding)
    raw_probabilities = raw_probabilities[0] # Get 1D array
    class_names = label_encoder.classes_

    # Scale to 0-100 and cast to native Python floats to prevent JSON serialization errors
    status_dict = {str(name): float(prob) * 100 for name, prob in zip(class_names, raw_probabilities)}

    # Calculate Deltas (Self-Improvement Metric)
    deltas = {}
    if deps.previous_state:
        for status in status_dict:
            deltas[status] = round(status_dict[status] - deps.previous_state.get(status, 0.0), 1)

    # Update Session State for the next turn
    deps.previous_state = {k: round(v, 1) for k, v in status_dict.items()}

    # Determine Crisis and Primary Tone
    suicidal_prob = status_dict.get('Suicidal', 0.0)
    is_crisis = bool(suicidal_prob >= 10.0) # 10% threshold
    primary_tone = max(status_dict, key=status_dict.get)

    return EmotionalState(
        is_crisis=is_crisis,
        primary_tone=primary_tone,
        suicidal_probability=round(suicidal_prob, 1),
        state_deltas=deltas,
        current_state={k: round(v, 1) for k, v in status_dict.items()}
    )

In [11]:
async def chat_with_agent(user_message: str, deps: MentalHealthDeps):
    print(f"\n👤 User: {user_message}")

    # Run the Agent
    result = await agent.run(user_message, deps=deps)

    # ✅ FIX: Automatically handle both new (.data) and old (.output) PydanticAI versions
    response = getattr(result, 'data', None) or result.output

    # Print internal reasoning (Great for debugging in Colab)
    print(f"🧠 [Internal Monologue]: {response.internal_monologue}")
    print(f"🛡️ [Safety Active]: {response.safety_override_active}")
    print(f"🤖 Agent: {response.message_to_user}")
    print("-" * 50)

    return response.message_to_user

# Initialize Dependencies (Empty previous_state for the first message)
deps = MentalHealthDeps()

print(" Starting Mental Health Companion (Groq + PydanticAI)")

# Simulate a conversation to test self-improvement (delta tracking)
test_messages = [
    "I'm so stressed about work, I can't breathe.",
    "I don't have the energy to do anything. It's just too much.",
    "Yeah, just sitting here. Thanks for listening."
]

for msg in test_messages:
    await chat_with_agent(msg, deps)

 Starting Mental Health Companion (Groq + PydanticAI)

👤 User: I'm so stressed about work, I can't breathe.
🧠 [Internal Monologue]: The user reports acute stress and feeling unable to breathe, indicating high stress (97) and moderate anxiety. No crisis flag. I should validate their feelings, express empathy, and gently suggest a grounding breathing technique while encouraging them to take a short break. Keep it concise (under 3 sentences).
🛡️ [Safety Active]: False
🤖 Agent: I’m really sorry you’re feeling so overwhelmed right now; it sounds incredibly stressful. If you can, try a simple breathing exercise: inhale slowly for four counts, hold for four, then exhale for six, and repeat a few times. Remember to give yourself a brief pause—it can help calm your body and mind.
--------------------------------------------------

👤 User: I don't have the energy to do anything. It's just too much.
🧠 [Internal Monologue]: The analysis indicates a crisis with suicidal probability 18.6% and high d